# 23. Three referee-closers, scoring only

1. **Wider guard panel.** Four more public encoder guards scored on every evaluation set used so far (BIPIA documents, NotInject, the second source, deepset, jailbreak) and reported next to the three released detectors and the structural models. A guard whose attack class is picked wrong shows up as AUROC below 0.5 on the direct set; its scores are inverted and the inversion is reported.
2. **Over-defense on instruction-bearing benign documents.** Real recipes or how-to text, which legitimately contain imperatives, scored by every detector; the fraction flagged at each detector's document-calibrated threshold and at 0.5.
3. **Injection-position sweep.** Every BIPIA injection re-embedded in its own host at the start, the middle and the end; AUROC against hosts and document-calibrated FNR per position, per detector.

GPU for scoring, Hugging Face token for Prompt-Guard. No training. Everything cached under `data/closers/`.

**Outputs.** `reports/guard_panel_extended.csv`, `reports/overdefense_howto.csv`, `reports/position_sweep.csv`, `figures/referee_closers.png`.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
!pip install -q transformers datasets accelerate sentencepiece
os.makedirs('reports', exist_ok=True); os.makedirs('figures', exist_ok=True); os.makedirs('data/closers', exist_ok=True)
import torch, gc; print('Session ready. GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

In [ ]:
!df -h /content/drive
!du -sh /content/drive/MyDrive/PICALIB_Research/picalib-research/data/structural/model_* 2>/dev/null
!ls -d /content/drive/MyDrive/PICALIB_Research/picalib-research/data/structural/model_*/checkpoint-* 2>/dev/null

In [ ]:
import numpy as np, glob, os
bad = []
for fp in sorted(glob.glob('data/**/*.npy', recursive=True)):
    try: np.load(fp, allow_pickle=True)
    except Exception as e: bad.append(fp); print('UNREADABLE', fp, '|', type(e).__name__, str(e)[:60])
print(f'{len(bad)} unreadable of {len(glob.glob("data/**/*.npy", recursive=True))} caches')
for fp in bad: os.remove(fp); print('removed', fp)

## Stage A. Module

In [ ]:
%%writefile src/referee_closers.py
"""Three scoring-only checks a referee will ask for: a wider guard panel, over-defense on benign documents that
legitimately carry instructions, and an injection-position sweep."""
from __future__ import annotations
import re
import numpy as np

# Additional public encoder guards. Each entry: tag -> list of candidate Hub ids (first that loads is used).
EXTRA_GUARDS = {
    "deepset_injection": ["deepset/deberta-v3-base-injection"],
    "protectai_v1":      ["protectai/deberta-v3-base-prompt-injection"],
    "piguard":           ["leolee99/PIGuard", "leolee99/InjecGuard"],
    "fmops_distilbert":  ["fmops/distilbert-prompt-injection"],
}

# Benign documents that legitimately contain imperatives (recipes, how-to). Candidate Hub datasets and text fields.
HOWTO_SOURCES = [
    ("corbt/all-recipes", "train", "input"),
    ("Hieu-Pham/kaggle_food_recipes", "train", "Instructions"),
    ("Shengtao/recipe", "train", "directions"),
]


def load_howto(n=500, min_chars=200, max_chars=1500, seed=0):
    """First candidate that loads; returns (texts, source_id) or ([], None)."""
    from datasets import load_dataset
    for cid, split, field in HOWTO_SOURCES:
        try:
            ds = load_dataset(cid, split=split)
            raw = ds[field]
            texts, seen = [], set()
            for t in raw:
                t = re.sub(r"\s+", " ", str(t if t is not None else "")).strip()
                if min_chars <= len(t) <= max_chars and t not in seen:
                    seen.add(t); texts.append(t)
            if len(texts) >= 100:
                rng = np.random.default_rng(seed)
                idx = rng.choice(len(texts), size=min(n, len(texts)), replace=False)
                return [texts[i] for i in sorted(idx)], cid
        except Exception:
            continue
    return [], None


def split_host_instruction(text):
    """BIPIA attack text is host + newline + injected instruction (last line)."""
    parts = text.rstrip().split("\n")
    return "\n".join(parts[:-1]).rstrip(), parts[-1].strip()


def _sentences(doc):
    parts = re.split(r"(?<=[a-z0-9]{2}[.!?])\s+(?=[\"'(A-Z])", doc.strip())
    return [p for p in parts if p]


def position_variants(text, rng):
    """start / middle / end placements of the same instruction in the same host, newline-delimited."""
    host, ins = split_host_instruction(text)
    out = {"end": host + "\n" + ins, "start": ins + "\n" + host}
    sents = _sentences(host)
    if len(sents) >= 3:
        k = rng.integers(1, len(sents))
        out["middle"] = " ".join(sents[:k]) + "\n" + ins + "\n" + " ".join(sents[k:])
    else:
        out["middle"] = out["end"]
    return out


def polarity_fix(p_direct, y_direct, scores_by_set):
    """If a guard ranks direct injections below benigns (AUROC < 0.5), its attack class was picked wrong: invert."""
    from sklearn.metrics import roc_auc_score
    a = roc_auc_score(y_direct, p_direct)
    if a < 0.5:
        return {k: 1.0 - v for k, v in scores_by_set.items()}, True, a
    return scores_by_set, False, a


## Stage B. Frames, existing caches, token

In [ ]:
import importlib, json, numpy as np, pandas as pd
import referee_closers, data_loaders, interval_panel; importlib.reload(referee_closers); importlib.reload(data_loaders); importlib.reload(interval_panel)
from referee_closers import EXTRA_GUARDS, load_howto, position_variants, polarity_fix
from data_loaders import load_deepset, load_bipia_categorized, load_notinject, load_jailbreak, BIPIA_HARMFUL, BIPIA_HIJACK, BIPIA_MALICIOUS
from detectors import score_released
from structural_train import score_texts
from interval_panel import three_thresholds, fnr, fpr
from sklearn.metrics import roc_auc_score
os.makedirs('data/closers', exist_ok=True)

deepset = load_deepset(); bip = load_bipia_categorized(per_category=30, categories=BIPIA_MALICIOUS); noti = load_notinject(); jb = load_jailbreak()
OPI = json.load(open('data/opi/opi_eval.json')); OTXT = [r['text'] for r in OPI['records']]; OY = np.array([r['label'] for r in OPI['records']])
yd = deepset.label.values; HOST = bip.label.values == 0; HIJ = bip.meta.isin(BIPIA_HIJACK).values; HARM = bip.meta.isin(BIPIA_HARMFUL).values
SETS = {'bipia': bip.text.tolist(), 'notinject': noti.text.tolist(), 'opi': OTXT, 'deepset': deepset.text.tolist(), 'jailbreak': jb.text.tolist()}

RELEASED = {'protectai_v2': 'protectai/deberta-v3-base-prompt-injection-v2', 'prompt_guard_2': 'meta-llama/Llama-Prompt-Guard-2-86M', 'prompt_guard_2_22m': 'meta-llama/Llama-Prompt-Guard-2-22M'}
LABEL = {'protectai_v2': 'ProtectAI-v2', 'prompt_guard_2': 'Prompt-Guard-2 (86M)', 'prompt_guard_2_22m': 'Prompt-Guard-2 (22M)', 'full': 'structural (full)', 'full_aug': 'structural (full_aug)',
         'deepset_injection': 'deepset injection', 'protectai_v1': 'ProtectAI-v1', 'piguard': 'PIGuard', 'fmops_distilbert': 'fmops distilbert'}
MDIR = {'full': 'data/structural/model_full/best', 'full_aug': 'data/structural/model_full_aug/best'}
from huggingface_hub import login; import getpass
_t = getpass.getpass('HF token (gated Prompt-Guard checkpoints): ').strip()
if _t: login(token=_t)
GUARD_ID = {}
def score_any(tag, texts):
    try:
        if tag in RELEASED: return score_released(texts, RELEASED[tag])
        if tag in MDIR: return score_texts(MDIR[tag], texts)
        return score_released(texts, GUARD_ID[tag])
    except Exception as e:
        raise SystemExit(f'scoring {tag} failed: {type(e).__name__}: {str(e)[:160]}\n'
                         'If this is a Prompt-Guard checkpoint, the token lacks access: re-run with a valid HF token. Completed scores are cached and will be skipped.')
def load_or_score(fp, tag, s):
    if not os.path.exists(fp): print('regenerating', fp); np.save(fp, score_any(tag, SETS[s]))
    a = np.load(fp); assert len(a) == len(SETS[s]), f'{fp}: {len(a)} vs {len(SETS[s])}'; return a
KEY = {'bipia': 'bipia_cat', 'notinject': 'notinject', 'deepset': 'deepset', 'jailbreak': 'jailbreak'}
S = {}
for t in RELEASED:
    S[t] = {s: load_or_score(f'data/score_{t}_{KEY[s]}.npy', t, s) for s in KEY}; S[t]['opi'] = load_or_score(f'data/opi/score_{t}.npy', t, 'opi')
for cfg in MDIR:
    S[cfg] = {s: load_or_score(f'data/structural/scores/{cfg}_{s}.npy', cfg, s) for s in ('bipia', 'notinject', 'deepset', 'jailbreak')}
    S[cfg]['opi'] = load_or_score(f'data/opi/score_structural_{cfg}.npy', cfg, 'opi')
print('existing scorers loaded:', list(S))

## Stage C. Closer 1: wider guard panel

In [ ]:
GUARD_ID = {}
for tag, cands in EXTRA_GUARDS.items():
    if all(os.path.exists(f'data/closers/score_{tag}_{s}.npy') for s in SETS):
        GUARD_ID[tag] = json.load(open(f'data/closers/id_{tag}.json')); S[tag] = {s: np.load(f'data/closers/score_{tag}_{s}.npy') for s in SETS}; print(tag, 'cached'); continue
    for cid in cands:
        try:
            print('trying', tag, cid); sc = {s: score_released(texts, cid) for s, texts in SETS.items()}
            GUARD_ID[tag] = cid; json.dump(cid, open(f'data/closers/id_{tag}.json', 'w'))
            for s, a in sc.items(): np.save(f'data/closers/score_{tag}_{s}.npy', a)
            S[tag] = sc; break
        except Exception as e:
            print(f'  {cid} failed: {type(e).__name__}: {str(e)[:120]}')
    if tag not in S: print(tag, 'unavailable; skipped')
FLIPPED = {}
for tag in list(EXTRA_GUARDS):
    if tag in S:
        S[tag], flipped, a = polarity_fix(S[tag]['deepset'], yd, S[tag]); FLIPPED[tag] = flipped
        if flipped: print(f'{tag}: attack class was inverted (direct AUROC {a:.3f}); scores flipped')
rows = []
for t in S:
    pb = S[t]['bipia']; hosts = pb[HOST]; tdoc = three_thresholds(hosts)['guar']; po = S[t]['opi']; to = three_thresholds(po[OY == 0])['guar']
    rows.append(dict(detector=LABEL.get(t, t), hub_id=RELEASED.get(t, GUARD_ID.get(t, MDIR.get(t, ''))), inverted=FLIPPED.get(t, False),
                     AUROC_hijack=roc_auc_score(np.r_[np.ones(HIJ.sum()), np.zeros(HOST.sum())], np.r_[pb[HIJ], hosts]),
                     AUROC_harmful=roc_auc_score(np.r_[np.ones(HARM.sum()), np.zeros(HOST.sum())], np.r_[pb[HARM], hosts]),
                     FNR_hijack_doc=fnr(pb[HIJ], tdoc), notinject_FPR_doc=fpr(S[t]['notinject'], tdoc), notinject_frac_over_0p5=float((S[t]['notinject'] >= 0.5).mean()),
                     AUROC_second_source=roc_auc_score(OY, po), FNR_second_source=fnr(po[OY == 1], to),
                     AUROC_direct=roc_auc_score(yd, S[t]['deepset']), AUROC_jailbreak=roc_auc_score(jb.label.values, S[t]['jailbreak'])))
PANEL = pd.DataFrame(rows); PANEL.to_csv('reports/guard_panel_extended.csv', index=False)
pd.set_option('display.width', 250); print('=== extended guard panel ==='); print(PANEL.drop(columns=['hub_id']).round(3).to_string(index=False))

## Stage D. Closer 2: over-defense on instruction-bearing benign documents

In [ ]:
hfp = 'data/closers/howto.json'
if os.path.exists(hfp): HOW = json.load(open(hfp)); print('how-to documents loaded from cache:', len(HOW['texts']), 'from', HOW['source'])
else:
    texts, src = load_howto(n=500); HOW = dict(texts=texts, source=src); json.dump(HOW, open(hfp, 'w')); print('how-to documents:', len(texts), 'from', src)
OD = None
if not HOW['texts']: print('no how-to source loaded; closer 2 skipped')
else:
    print('example:', HOW['texts'][0][:250])
    rows = []
    for t in S:
        fp = f'data/closers/score_{t}_howto.npy'
        if not os.path.exists(fp): print('scoring', t, 'on how-to'); np.save(fp, score_any(t, HOW['texts']))
        p = np.load(fp); p = 1.0 - p if FLIPPED.get(t, False) else p
        tdoc = three_thresholds(S[t]['bipia'][HOST])['guar']
        rows.append(dict(detector=LABEL.get(t, t), n=len(p), source=HOW['source'], howto_FPR_doc_threshold=fpr(p, tdoc), howto_frac_over_0p5=float((p >= 0.5).mean()),
                         bipia_host_FPR_doc_threshold=fpr(S[t]['bipia'][HOST], tdoc), notinject_FPR_doc_threshold=fpr(S[t]['notinject'], tdoc)))
    OD = pd.DataFrame(rows); OD.to_csv('reports/overdefense_howto.csv', index=False)
    print('\n=== over-defense on benign how-to documents (imperatives that are content) ==='); print(OD.round(3).to_string(index=False))

## Stage E. Closer 3: injection-position sweep on BIPIA

In [ ]:
pfp = 'data/closers/positions.json'
if os.path.exists(pfp): POS = json.load(open(pfp)); print('position variants loaded from cache')
else:
    rng = np.random.default_rng(0); atk_idx = np.where(bip.label.values == 1)[0]
    POS = {'idx': atk_idx.tolist(), 'start': [], 'middle': [], 'end': []}
    for i in atk_idx:
        v = position_variants(bip.text.iloc[i], rng)
        for k in ('start', 'middle', 'end'): POS[k].append(v[k])
    json.dump(POS, open(pfp, 'w'))
atk_idx = np.array(POS['idx']); is_hij = HIJ[atk_idx]; is_harm = HARM[atk_idx]
rows = []
for t in S:
    hosts = S[t]['bipia'][HOST]; tdoc = three_thresholds(hosts)['guar']
    for k in ('start', 'middle', 'end'):
        fp = f'data/closers/score_{t}_pos_{k}.npy'
        if not os.path.exists(fp): print('scoring', t, k); np.save(fp, score_any(t, POS[k]))
        p = np.load(fp); p = 1.0 - p if FLIPPED.get(t, False) else p
        for tier, m in (('hijack', is_hij), ('harmful', is_harm)):
            rows.append(dict(detector=LABEL.get(t, t), position=k, tier=tier, n=int(m.sum()),
                             AUROC=roc_auc_score(np.r_[np.ones(m.sum()), np.zeros(HOST.sum())], np.r_[p[m], hosts]), FNR_doc=fnr(p[m], tdoc)))
PS = pd.DataFrame(rows); PS.to_csv('reports/position_sweep.csv', index=False)
print('=== position sweep: AUROC vs hosts ==='); print(PS.pivot_table(index=['detector', 'tier'], columns='position', values='AUROC')[['start', 'middle', 'end']].round(3).to_string())
print('\n=== position sweep: FNR at document-calibrated threshold ==='); print(PS.pivot_table(index=['detector', 'tier'], columns='position', values='FNR_doc')[['start', 'middle', 'end']].round(3).to_string())

## Stage F. Figure

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
d = PANEL.sort_values('AUROC_hijack', ascending=False); assert len(d)
axes[0].barh(d.detector, d.AUROC_hijack); axes[0].axvline(0.5, color='k', ls=':', lw=.8); axes[0].set_xlim(0, 1); axes[0].set_title('Hijack AUROC vs host documents'); axes[0].invert_yaxis(); axes[0].grid(axis='x', alpha=.3)
if OD is not None:
    o = OD.set_index('detector').loc[d.detector]; axes[1].barh(o.index, o.howto_FPR_doc_threshold, label='how-to benigns'); axes[1].barh(o.index, o.notinject_FPR_doc_threshold, alpha=.5, label='NotInject')
    axes[1].set_xlim(0, 1); axes[1].set_title('Over-defense at document-calibrated threshold'); axes[1].invert_yaxis(); axes[1].legend(fontsize=8); axes[1].grid(axis='x', alpha=.3)
ph = PS[PS.tier == 'hijack'].pivot_table(index='detector', columns='position', values='FNR_doc')[['start', 'middle', 'end']].loc[d.detector]; assert len(ph)
ph.plot.bar(ax=axes[2], width=.8); axes[2].set_ylim(0, 1.05); axes[2].set_ylabel('hijack FNR at document threshold'); axes[2].set_title('Injection position'); axes[2].legend(fontsize=8); axes[2].grid(axis='y', alpha=.3); axes[2].tick_params(axis='x', labelrotation=30, labelsize=7)
plt.tight_layout(); plt.savefig('figures/referee_closers.png', dpi=150); plt.show()

## Stage G. Log and commit

In [ ]:
from reslog import log_result
summary = ('Extended panel:\n' + PANEL.drop(columns=['hub_id']).round(3).to_string(index=False)
           + ('\n\nOver-defense (how-to):\n' + OD.round(3).to_string(index=False) if OD is not None else '\n\n(how-to over-defense skipped)')
           + '\n\nPosition sweep (hijack FNR_doc):\n' + PS[PS.tier == 'hijack'].pivot_table(index='detector', columns='position', values='FNR_doc')[['start', 'middle', 'end']].round(3).to_string())
log_result('nb23: referee closers: extended guard panel, how-to over-defense, position sweep', summary, csv_df=PANEL, csv_name='guard_panel_extended.csv')
print(summary)

# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
                    "nb23: extended guard panel, over-defense on how-to documents, injection-position sweep; add src/referee_closers.py"],
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr)